### Instalacja biliotek

In [8]:
#pip install -r requirements.txt

### Model

In [9]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("OPEN_AI_KEY")

client = OpenAI(
    api_key=api_key,
)

In [ ]:
import ipywidgets as widgets
from utils.AudioManager import VoiceRecorder

# Widget do outputu
output_area = widgets.Output()
display(output_area)

# Definiujemy model w main
MODEL_NAME = "whisper-1" 
wavPath = r"C:\Users\szymo\Desktop\studia\mall-center-ai-assistant\voice\input.wav"
textPath = r"C:\Users\szymo\Desktop\studia\mall-center-ai-assistant\voice\transcription.txt"
# Przekazujemy model do klasy
recorder = VoiceRecorder(output_area=output_area,openai_client=client,wav_path=wavPath, txt_path=textPath)

# Przyciski
btn_start = widgets.Button(description="Start", icon="microphone")
btn_stop = widgets.Button(description="Stop", icon="stop")
btn_transcribe = widgets.Button(description="Transkrybuj", icon="file-text")

btn_start.on_click(lambda x: recorder.start_recording())
btn_stop.on_click(lambda x: recorder.stop_recording())
btn_transcribe.on_click(lambda x: recorder.transcribe())

display(widgets.HBox([btn_start, btn_stop, btn_transcribe]))

Output()

### Połączenie kategorii z danym sklepem

In [11]:
import pandas as pd
from utils.ShopAssistant import ShopAssistant

df = pd.read_csv("./db/shops.csv", sep=";")
df.head()

,sklep,kategorie
0,ikea,"meble,biuro,kuchnia,narzedzia"
1,media_markt,"elektronika,agd,rtv"
2,zalando,"moda,obuwie,akcesoria"
3,castorama,"budowa,ogrod,narzedzia"
4,empik,"ksiazki,muzyka,gry"


In [ ]:
assistant = ShopAssistant(df, client)

with open("./voice/transcription.txt", "r", encoding="utf-8") as f:
    text = f.read()

result = assistant.analyze_intent(question=text)
print("Analiza intencji:", result)

Asystent załadowany. Znaleziono 15 unikalnych kategorii.
Analiza intencji: {'input_text': 'Chciałbym kupić młotek.', 'detected_categories': ['narzedzia'], 'matching_shops': ['ikea', 'castorama']}


### Pobranie odpowiednich produktow

In [13]:
df_products = pd.read_csv("./db/products.csv", sep=";")
df_products.head()

target_shops = result['matching_shops']
shop_inventory = df_products[df_products['sklep'].isin(target_shops)]
print(f"--- Asortyment w sklepach: {target_shops} ---")
display(shop_inventory)

--- Asortyment w sklepach: ['ikea', 'castorama'] ---


,sklep,produkt
0,ikea,biurko
1,ikea,krzesło
2,ikea,szafka_kuchenna
3,ikea,zestaw_narzedzi
12,castorama,mlotek
13,castorama,wiertarka
14,castorama,kosiarka
15,castorama,farba_scienna


### Przekazanie produktow modelowi

In [21]:
system_prompt = """
Jesteś asystentem analizującym bazę danych produktów dostępnych w sklepie.
użytkownik zapyta Cię o dostępność określonych produktów lub powie ci cel wizyty w naszej galerii handlowej.
otrzymasz listę sklepów, które mogą spełniać jego potrzeby.
Twoim zadaniem jest wygenerowanie przyjaznej odpowiedzi, która podsumowuje te informacje i zachęca użytkownika do odwiedzenia tych sklepów.
Pamiętaj, aby odpowiedź była zwięzła i uprzejma.
Jeżeli nie ma pasujących sklepów, uprzejmie poinformuj użytkownika, że nie znaleziono odpowiednich opcji.
Nie dokładaj żadnych dodatkowych informacji ani nie sugeruj innych sklepów, cen czy godzin otwarcia bo na to danych nie posiadamy.
"""

In [22]:
query = text + "\nNa podstawie powyższego zapytania oraz listy sklepów: " + ", ".join(target_shops) + ", wygeneruj przyjazną odpowiedź dla użytkownika."
messages=[
            {"role": "developer", "content": system_prompt},        
            {"role": "user", "content": query}
]

In [23]:
response = client.responses.create(
    model="gpt-5-mini",
    input=messages
)

In [24]:
print(response.output_text)

Świetnie — jeśli szukasz młotka, możesz sprawdzić następujące sklepy: Castorama oraz IKEA. Zachęcam do odwiedzenia ich i życzę udanych zakupów!


### Odpowiedz głosowa

In [26]:
from IPython.display import Audio
audio_response = client.audio.speech.create(
    model="tts-1",  
    voice="alloy",
    input=response.output_text,
    speed=1.2
)
audio_bytes = audio_response.read()
Audio(audio_bytes, autoplay=True)